# Содержание
### 1-6: построение графа
1. Стереть Neo4j базу до нуля (7688)
2. Вызов DeepSeek с промптом извлечения узлов и связей из сообщения (cклеиваем строки в промпт (+ сбщ, + онто))
3. JSON -> Cypher конвертер (LLM возвращает JSON с узлами и нодами, чтобы в конт окно влезало. Cypher не влезал!) ВАЖНО: он зависит от текущей онтологии. если та поменялась, его тоже меняем
4. Запись в файл cypher_checkpoints - слепки базы на каждое сбщ
5. Внесение инфы в базу (исполнение всех Cypher команд по очереди)
6. Повторить 2-6 пункты для всех сбщ

### 7-N: RAG
   

In [ ]:
# 0 инит ключей

In [67]:
%pip -q install python-dotenv

from dotenv import load_dotenv
import os

load_dotenv(".env")   # подхватит /.../neo4j_граф/.env

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
print("DEEPSEEK_API_KEY loaded:", bool(DEEPSEEK_API_KEY))

Note: you may need to restart the kernel to use updated packages.
DEEPSEEK_API_KEY loaded: True


In [13]:
# 1 стирание базы полное до нуля

In [63]:
def step_1_reset_db():
    !docker stop neo4j_travel || true
    !docker rm neo4j_travel || true
    !docker volume rm neo4j_travel_data || true
    
    !docker run -d --name neo4j_travel \
      -p 7475:7474 -p 7688:7687 \
      -e NEO4J_AUTH=neo4j/12345678 \
      -v neo4j_travel_data:/data \
      neo4j:5
    
    # !docker ps --filter name=neo4j_travel   
    print("Did reset DB")
    return None


neo4j_travel
neo4j_travel
neo4j_travel_data
fc42c2acda3dd862edb1762ad1f2d1e7012b58436ca502402de952e9f2806a21
Did reset DB


In [33]:
# 1 стирание второй базы

def step_1_reset_db_2():
    !docker stop neo4j_travel2 || true
    !docker rm neo4j_travel2 || true
    !docker volume rm neo4j_travel2_data || true

    !docker run -d --name neo4j_travel2 \
      -p 7476:7474 -p 7689:7687 \
      -e NEO4J_AUTH=neo4j/12345678 \
      -v neo4j_travel2_data:/data \
      neo4j:5

    print("Did reset DB #2 (http://localhost:7476, bolt neo4j://localhost:7689)")
    return None

In [66]:
# 2 сборка промпта и запрос к DeepSeek - выход ABox в виде JSON

In [16]:
def step_2_prompt_to_abox(msg_as_JSON) -> str:
    import os
    import json
    import requests
    import time

    PROMPT_PATH = "prompt.txt"
    ONTO_PATH = "ontology.txt"

    DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
    if not DEEPSEEK_API_KEY:
        raise RuntimeError("Set env var DEEPSEEK_API_KEY (export DEEPSEEK_API_KEY=...).")

    DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com").rstrip("/")
    MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-reasoner")  # or deepseek-chat

    # --- load prompt template + ontology ---
    with open(PROMPT_PATH, "r", encoding="utf-8") as f:
        prompt_template = f.read()

    with open(ONTO_PATH, "r", encoding="utf-8") as f:
        ontology_text = f.read().strip()

    # --- normalize message JSON string ---
    if isinstance(msg_as_JSON, (dict, list)):
        message_json_string = json.dumps(msg_as_JSON, ensure_ascii=False)
    else:
        message_json_string = str(msg_as_JSON).strip()

    # validate that it's JSON (LLM expects JSON string with keys post_id/post_url/author/date/text)
    try:
        json.loads(message_json_string)
    except Exception as e:
        raise ValueError(f"msg_as_JSON must be a valid JSON string (or dict). Parse error: {e}") from e

    # --- stitch prompt ---
    prompt = (
        prompt_template
        .replace("{RDF_ONTOLOGY_HERE}", ontology_text)
        .replace("{MESSAGE_JSON_STRING_HERE}", message_json_string)
    )

    # sanity check for unresolved placeholders
    unresolved = [p for p in ["{RDF_ONTOLOGY_HERE}", "{MESSAGE_JSON_STRING_HERE}"] if p in prompt]
    if unresolved:
        raise RuntimeError(f"Unresolved placeholders in prompt: {unresolved}")

    # --- call DeepSeek ---
    url = f"{DEEPSEEK_BASE_URL}/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": MODEL,
        "messages": [
            # system можно оставить очень коротким, промпт уже строгий
            {"role": "system", "content": "Return ONLY valid JSON object. No markdown. No code fences."},
            {"role": "user", "content": prompt},
        ],
        "temperature": 0.0,
    }

    t0 = time.perf_counter()
    resp = requests.post(url, headers=headers, json=payload, timeout=180)
    elapsed = time.perf_counter() - t0
    print(f"DeepSeek API latency: {elapsed:.2f}s")
    resp.raise_for_status()
    data = resp.json()

    api_finish = data["choices"][0].get("finish_reason", "")
    content = data["choices"][0]["message"]["content"]

    emoji = "🟢" if api_finish != "length" else "🔴"
    print(f"{emoji} API finish_reason = {api_finish}")

    # optional: validate JSON (hard fail if model returned junk)
    try:
        json.loads(content)
    except Exception as e:
        raise ValueError(f"Model did not return valid JSON. Error: {e}\nRaw content:\n{content[:2000]}") from e

    print(content)  # preview
    return content


In [20]:
# 3 JSON → Cypher генерация

In [24]:
def step_3_abox_to_cypher(LLM_answer: str) -> List[str]:
    import json
    import re

    # =========================
    # Helpers
    # =========================

    def strip_code_fences(s: str) -> str:
        s = (s or "").strip()
        if s.startswith("```"):
            s = re.sub(r"^```(?:json)?\s*", "", s)
            s = re.sub(r"\s*```$", "", s)
        return s.strip()

    def neo4j_local_name(name: str) -> str:
        if not name:
            return ""
        if ":" in name:
            return name.split(":")[-1]
        if "#" in name:
            return name.split("#")[-1]
        if "/" in name:
            return name.rsplit("/", 1)[-1]
        return name

    def neo4j_safe_ident(name: str) -> str:
        n = neo4j_local_name(name)
        n = re.sub(r"[^A-Za-z0-9_]", "_", n)
        if not n:
            return "X"
        if n[0].isdigit():
            n = "_" + n
        return n

    def cypher_literal(v, prop_name=None):
        if v is None:
            return "null"
        if isinstance(v, bool):
            return "true" if v else "false"
        if isinstance(v, (int, float)):
            return str(v)

        # New ontology/prompt: datetime is stored as ISO8601 string in pcfr:sourcePostDateTime
        if prop_name in {"sourcePostDateTime"}:
            # Expecting something like "2013-02-06T06:19:00"
            return f"localdatetime('{str(v)}')"

        s = str(v)
        s = s.replace("\\", "\\\\").replace("'", "\\'")
        s = s.replace("\n", "\\n").replace("\r", "\\r")
        return f"'{s}'"

    def contains_cyrillic(s: str) -> bool:
        return bool(re.search(r"[А-Яа-яЁё]", s or ""))

    # =========================
    # Parse JSON from LLM
    # =========================

    raw = strip_code_fences(LLM_answer)
    root = json.loads(raw)

    payload = root.get("payload", {}) or {}
    nodes = payload.get("nodes", []) or []
    rels = payload.get("relationships", []) or []

    cypher_statements = []

    # =========================
    # Constraints
    # =========================

    labels = sorted({
        neo4j_safe_ident(n.get("label", ""))
        for n in nodes
        if n.get("label")
    })

    for label in labels:
        cypher_statements.append(
            f"CREATE CONSTRAINT IF NOT EXISTS FOR (n:{label}) REQUIRE n.key IS UNIQUE"
        )

    # =========================
    # Nodes
    # =========================

    for node in nodes:
        label_raw = node.get("label", "")
        key = node.get("key")

        if not label_raw or key is None:
            continue

        label = neo4j_safe_ident(label_raw)
        props = node.get("properties", {}) or {}

        set_parts = []

        for k, v in props.items():
            prop_key = neo4j_safe_ident(k)

            # New prompt: output should be Latin-only, but keep a warning just in case.
            if isinstance(v, str) and contains_cyrillic(v):
                print(f"⚠ WARNING: Cyrillic detected in property '{prop_key}' → '{v}'")

            set_parts.append(f"n.{prop_key}={cypher_literal(v, prop_key)}")

        set_clause = (" SET " + ", ".join(set_parts)) if set_parts else ""
        stmt = f"MERGE (n:{label} {{key:{cypher_literal(key)}}}){set_clause}"
        cypher_statements.append(stmt)

    # =========================
    # Relationships
    # =========================

    for rel in rels:
        f = rel.get("from", {}) or {}
        t = rel.get("to", {}) or {}

        fl_raw = f.get("label", "")
        fk = f.get("key")
        tl_raw = t.get("label", "")
        tk = t.get("key")
        rt_raw = rel.get("type", "")

        if not fl_raw or not tl_raw or not rt_raw or fk is None or tk is None:
            continue

        fl = neo4j_safe_ident(fl_raw)
        tl = neo4j_safe_ident(tl_raw)
        rt = neo4j_safe_ident(rt_raw)

        stmt = (
            f"MATCH (a:{fl} {{key:{cypher_literal(fk)}}}), "
            f"(b:{tl} {{key:{cypher_literal(tk)}}}) "
            f"MERGE (a)-[:{rt}]->(b)"
        )
        cypher_statements.append(stmt)

    # =========================
    # Output
    # =========================

    print("LLM finish_reason:", root.get("finish_reason"))
    print("LLM reason:", root.get("reason"))
    print(f"Prepared {len(cypher_statements)} Cypher statements.\n")
    return cypher_statements

In [25]:
# 4 Сохранение Cypher в чекпоинт

In [26]:
def step_4_save_cypher(cypher_statements: List[str]):
    import os
    import re
    
    ckpt_dir = "cypher_checkpoints"
    os.makedirs(ckpt_dir, exist_ok=True)
    
    existing = []
    for name in os.listdir(ckpt_dir):
        m = re.match(r"^cypher_(\d+)\.txt$", name)
        if m:
            existing.append(int(m.group(1)))
    
    next_num = max(existing, default=0) + 1
    out_path = os.path.join(ckpt_dir, f"cypher_{next_num}.txt")
    
    with open(out_path, "w", encoding="utf-8") as f:
        for stmt in cypher_statements:
            f.write(stmt)
            f.write(";")
    
    print(f"Saved {len(cypher_statements)} statements to {out_path}")
    return None


In [28]:
# 5 исполнение Cypher (каждая строка автономна)

In [29]:
def step_5_execute_cypher(cypher_statements: List[str]):
    from neo4j import GraphDatabase
    
    URI = "neo4j://localhost:7688"
    USER = "neo4j"
    PASSWORD = "12345678"
    DATABASE = "neo4j"
    
    driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
    
    try:
        with driver.session(database=DATABASE) as session:
            for i, stmt in enumerate(cypher_statements, 1):
                session.run(stmt).consume()
        print(f"✅ Executed {len(cypher_statements)} statements.")
    finally:
        driver.close()    
    return None


In [ ]:
# 6 запуск пайплайна построения графа

In [31]:
step_1_reset_db()

import json

with open("messages/messages_fr_1.json", "r", encoding="utf-8") as f:
    messages = json.load(f)
messages = list(map(lambda x: json.dumps(x, ensure_ascii=False), messages))


for msg in messages:
    print(msg)
    answer = step_2_prompt_to_abox(msg) 
    cypher_statements = step_3_abox_to_cypher(answer)
    step_4_save_cypher(cypher_statements)
    step_5_execute_cypher(cypher_statements)

# subprocess.run(["afplay", "/System/Library/Sounds/Blow.aiff"], check=False)


neo4j_travel
neo4j_travel
neo4j_travel_data
953258c578b422700a1544dbfefc5cfc37b7e87da2c9902189cb6be41e52047d
Did reset DB
{"post_id": 3902451, "post_url": "https://forum.awd.ru/viewtopic.php?p=3902451#p3902451", "author": "!Casper!", "date": "11 июн 2013, 09:05", "text": "Прилетали вдвоем с другом в Париж, ШДГ, друг прошел контроль за 10 сек максиму без каких либо вопросов, а вот мне устроили допрос минут на 5. Попросили показать обратный билет, показать страховку, бронь отеля, спросили оплачен ли отель, показать кредитные карты, наличные деньги, спросили сколько наличности при себе, спросили какой картой я оплачивал отель, цель визита, на сколько дней приехал, кем я работаю в России. После этого поставили штамп в паспорт! Вывод: держите все распечатки билетов, броней отеля, страховки и тд при себе, так как могут возникнуть проблемы!"}
DeepSeek API latency: 323.83s
🟢 API finish_reason = stop
{
  "payload": {
    "ontology_version": "pcfr-v4",
    "nodes": [
      {
        "label": "Ci

In [226]:
# 7 простой RAG

In [242]:
# 7.1 Generate schema + Cypher (wrapped)
from neo4j import GraphDatabase
import os
import requests
import json

def rag_gen_schema_cypher(user_query: str):
    # -----------------------------
    # Config
    # -----------------------------
    URI = "neo4j://localhost:7689"
    USER = "neo4j"
    PASSWORD = "12345678"
    DATABASE = "neo4j"

    DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
    if not DEEPSEEK_API_KEY:
        raise RuntimeError("Set env var DEEPSEEK_API_KEY first.")

    DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com").rstrip("/")
    MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-reasoner")

    url = f"{DEEPSEEK_BASE_URL}/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json",
    }

    # -----------------------------
    # Helper: call DeepSeek
    # -----------------------------
    def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.0) -> str:
        payload = {
            "model": MODEL,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            "temperature": temperature,
        }

        resp = requests.post(url, headers=headers, json=payload, timeout=180)
        resp.raise_for_status()
        data = resp.json()
        return data["choices"][0]["message"]["content"]

    # =========================================================
    # 0️⃣ Extract DB schema (real schema injection)
    # =========================================================

    driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

    try:
        with driver.session(database=DATABASE) as session:

            labels = session.run(
                "CALL db.labels() YIELD label RETURN collect(label) AS labels"
            ).single()["labels"]

            rel_types = session.run(
                "CALL db.relationshipTypes() YIELD relationshipType "
                "RETURN collect(relationshipType) AS relTypes"
            ).single()["relTypes"]

            label_props = {}
            for label in labels:
                result = session.run(
                    f"MATCH (n:{label}) RETURN keys(n) AS keys LIMIT 1"
                ).single()
                label_props[label] = result["keys"] if result else []

    finally:
        driver.close()

    schema_text = (
        f"Available node labels: {labels}"
        f"Available relationship types: {rel_types}"
        f"Node properties by label: {json.dumps(label_props)}"
    )

    # =========================================================
    # 1️⃣ Generate Cypher (STRICT schema-aware prompt)
    # =========================================================

    system_cypher = """
You are a Neo4j Cypher generator.

INPUT YOU WILL RECEIVE:
- Database schema text (labels, relationship types, property keys).
- A user question.

HARD RULES (MUST FOLLOW):
1) Use ONLY the labels, relationship types, and property keys present in the provided schema text.
2) Do NOT invent any labels, relationship types, or properties.
3) Read-only queries only: MATCH / OPTIONAL MATCH / WHERE / WITH / RETURN.
   - Do NOT use CREATE, MERGE, DELETE, SET, CALL.
4) Always include LIMIT.
   - Use LIMIT 50 for list outputs.
   - Use LIMIT 1 for aggregated outputs.
5) If filtering by real-world entities (France, Paris, CDG),
   use entityName or key only if visible in schema sample.
   Never guess key format.
6) Be robust:
   - Use OPTIONAL MATCH when appropriate.
   - Use DISTINCT to avoid double counting.
7) If questioned about country, check all the cities of the country to give nodes and links
FREQUENCY RULE (CRITICAL):
If the question asks about frequency / how often / percentage:
- Compute:
    total_count   = total relevant Encounter in scope
    match_count   = number satisfying condition
    share         = toFloat(match_count) / total_count
- Return ALL THREE fields.

OUTPUT FORMAT:
Return ONLY valid JSON:
{"cypher": "<ONE Cypher query>"}
No markdown. No explanation.
"""

    prompt_cypher = (
        f"Database schema: {schema_text}"
        f"User question: {user_query}"
        "Generate Cypher query."
    )

    cypher_json = call_llm(system_cypher, prompt_cypher, temperature=0.0)

    try:
        cypher_obj = json.loads(cypher_json)
        cypher = cypher_obj.get("cypher", "").strip()
    except Exception:
        cypher = ""

    return schema_text, cypher, call_llm


In [243]:
# 7.2 Execute Cypher (wrapped, with error capture)
from neo4j import GraphDatabase
import json

def rag_execute_cypher(schema_text: str, cypher: str):
    URI = "neo4j://localhost:7689"
    USER = "neo4j"
    PASSWORD = "12345678"
    DATABASE = "neo4j"

    def split_statements(text):
        stmts = []
        buf = []
        in_single = False
        in_double = False
        for ch in text:
            if ch == "'" and not in_double:
                in_single = not in_single
            elif ch == '"' and not in_single:
                in_double = not in_double
            if ch == ';' and not in_single and not in_double:
                stmt = ''.join(buf).strip()
                if stmt:
                    stmts.append(stmt)
                buf = []
            else:
                buf.append(ch)
        last = ''.join(buf).strip()
        if last:
            stmts.append(last)
        return stmts

    facts = []
    exec_error = False
    exec_error_text = ''

    if cypher:
        driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
        try:
            with driver.session(database=DATABASE) as session:
                statements = split_statements(cypher)
                total = len(statements)
                for i, stmt in enumerate(statements, 1):
                    if i == total:
                        records = session.run(stmt).data()
                        facts.extend(records)
                    else:
                        session.run(stmt).consume()
        except Exception as e:
            exec_error = True
            exec_error_text = str(e)
            print(exec_error_text)
        finally:
            driver.close()

    facts_text = json.dumps(facts, ensure_ascii=False)
    return facts_text, exec_error, exec_error_text


In [244]:
# 7.2b Fix Cypher on error (wrapped)
from pathlib import Path

def rag_fix_cypher_on_error(user_query: str, schema_text: str, cypher: str, exec_error: bool, exec_error_text: str, call_llm):
    if not exec_error:
        return cypher

    prompt_path = Path('/Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/prompt_2_reviewer.txt')
    prompt_2_template = prompt_path.read_text(encoding='utf-8')

    prompt_2 = (
        prompt_2_template
        .replace('{QUESTION}', user_query)
        .replace('{SCHEMA}', schema_text)
        .replace('{CYPHER}', cypher)
        .replace('{ERROR}', exec_error_text)
    )

    fixed_cypher = call_llm("", prompt_2, temperature=0.0).strip()
    print('ERROR')
    print('fixed cypher:')
    print(fixed_cypher)
    return fixed_cypher


In [245]:
# 7.3 Generate final answer and print (wrapped)

def rag_answer_from_facts(
    user_query: str,
    schema_text: str,
    cypher: str,
    facts_text: str,
    call_llm,
):
    system_answer = (
        "You are a QA assistant that answers a user question using the result of a Cypher query. "
        "INPUTS YOU RECEIVE: Question, Cypher, Facts (JSON array). "
        "RULES: "
        "1) Use ONLY the provided Facts; no outside knowledge. "
        "2) If Facts is empty, answer: 'I do not know based on the database facts.' "
        "3) If Facts contain triple-like columns (s/p/o or s_label/s_key/p/o_label/o_key), interpret them as graph triples. "
        "4) If any field name ends with '.documentType' OR equals 'documentType', treat those values as document types and list unique values. "
        "5) If fields total_count/match_count/share exist, interpret them as frequency stats. "
        "6) Otherwise, summarize unique values most relevant to the question. "
        "OUTPUT: concise direct answer, no JSON, no citations."
    )

    prompt_answer = (
        f"Question:\n{user_query}\n\n"
        f"Cypher:\n{cypher}\n\n"
        f"Facts:\n{facts_text}\n\n"
        "Answer:"
    )

    final_answer = call_llm(system_answer, prompt_answer, temperature=0.2)

    print("\n--- Injected Schema ---")
    print(schema_text)

    print("\n--- Generated Cypher ---")
    print(cypher)

    print("\n--- Retrieved Facts ---")
    print(facts_text)

    print("\n--- Final Answer ---")
    print(final_answer)

In [8]:
# 8 Проверка RAG

In [257]:
# 8 Проверка RAG
# user_query = "What countries the dataset has". 
# user_query = "Which documents do I need to bring to passport control in France?"
# user_query = "Does passport control in Paris need many documents?"


user_query = "Are travelers often checked for return tickets?" # хорошо
# OPT + SHIFT + DOWN
# rag_answer('Is passport control in Paris strict?')
# rag_answer('Is passport control in Paris takes much time?')
# rag_answer('How often is return ticket required on passport control in France?')
# rag_answer('Give me IDs of messages about encounters where return ticket was required on passport control in France')
# rag_answer('What countries dataset has?')


schema_text, cypher, call_llm = rag_gen_schema_cypher(user_query)

# 🟥 TESTING
# cypher = 'MATCH (c:Country) RETURN p.entityName, p.key LIMIT 50'
# 🟥 TESTING
facts_text, exec_error, exec_error_text = rag_execute_cypher(schema_text, cypher)
cypher = rag_fix_cypher_on_error(user_query, schema_text, cypher, exec_error, exec_error_text, call_llm)
# Re-run if fixed
if exec_error:
    facts_text, exec_error, exec_error_text = rag_execute_cypher(schema_text, cypher)

rag_answer_from_facts(user_query, schema_text, cypher, facts_text, call_llm)
import subprocess
subprocess.run(["afplay", "/System/Library/Sounds/Blow.aiff"], check=False)



--- Injected Schema ---
Available node labels: ['City', 'Country', 'DocumentCheck', 'DocumentInstance', 'Encounter', 'Traveller', 'Airport', 'Questioning', 'BiometricCheck', 'Stamping', 'BorderOfficer', 'AtDesk', 'Station']Available relationship types: ['hasTraveler', 'atCity', 'atCountry', 'hasDocumentCheck', 'requestedDocument', 'locatedInCountry', 'atAirport', 'hasQuestioning', 'locatedInCity', 'hasBiometricCheck', 'hasStamping', 'stampsDocument', 'hasOfficer', 'issuedBy', 'atStation', 'locatedInCityStation']Node properties by label: {"City": ["key", "entityName"], "Country": ["key", "entityName"], "DocumentCheck": ["key"], "DocumentInstance": ["documentType", "key"], "Encounter": ["key", "sourcePostDateTime", "sourcePostID", "sourcePostURL", "sourcePostAuthor"], "Traveller": ["key"], "Airport": ["key", "entityName"], "Questioning": ["key", "topicKey", "wasAsked"], "BiometricCheck": ["biometricWasTaken", "biometricModality", "key"], "Stamping": ["key"], "BorderOfficer": ["key"], "A

CompletedProcess(args=['afplay', '/System/Library/Sounds/Blow.aiff'], returncode=0)

# Пайплайн вычисления метрик по датасету (вопрос; триплеты)
# По вопросу создать cypher, сгенерировать JSON с триплетами из него, сравнить их и gold и посчитать hits@k 

ВАЖНО: в базе ровно 1 сбщ ШДГ.

1. на диске: gold_dataset_1.json - файл с моими проверенными глазами триплетами по 1 сообщению ШДГ
2. LLM -> Cypher (особая выдача указана в промпте) -> JSON c ключами s p o -> приведение триплетов к виду строк
3. Приведение золотых триплетов к виду строк и вычисление Метрик Hits@k

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "requests", "charset-normalizer"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "neo4j"])


In [32]:
# 10  вопрос → (schema inject) → DeepSeek → Cypher (JSON) по вопросу

In [33]:
# === Cell 1: Ask DeepSeek to generate Cypher that returns triples ===
user_query = "Which documents do I need to bring to passport control in Paris?"

import os, json, re, requests
from pathlib import Path
from neo4j import GraphDatabase

# ---- Neo4j config ----
URI = "neo4j://localhost:7689"
USER = "neo4j"
PASSWORD = "12345678"
DATABASE = "neo4j"

# ---- DeepSeek config ----
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
if not DEEPSEEK_API_KEY:
    raise RuntimeError("DEEPSEEK_API_KEY is not set (load .env first).")

DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com").rstrip("/")
MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-reasoner")

# ---- Load prompt template ----
prompt_tpl = Path("prompt_retrieval.txt").read_text(encoding="utf-8")

# ---- Extract DB schema text ----
driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
try:
    with driver.session(database=DATABASE) as session:
        labels = session.run("CALL db.labels() YIELD label RETURN collect(label) AS labels").single()["labels"]
        rel_types = session.run(
            "CALL db.relationshipTypes() YIELD relationshipType RETURN collect(relationshipType) AS relTypes"
        ).single()["relTypes"]

        label_props = {}
        for label in labels:
            r = session.run(f"MATCH (n:{label}) RETURN keys(n) AS keys LIMIT 1").single()
            label_props[label] = r["keys"] if r else []
finally:
    driver.close()

schema_text = (
    f"Available node labels: {labels}"
    f"Available relationship types: {rel_types}"
    f"Node properties by label: {json.dumps(label_props, ensure_ascii=False)}"
)

# ---- Fill prompt in memory (do not modify file) ----
prompt = (
    prompt_tpl
    .replace("{SCHEMA_TEXT}", schema_text)
    .replace("{USER_QUERY}", user_query)
)

# ---- Call DeepSeek ----
url = f"{DEEPSEEK_BASE_URL}/v1/chat/completions"
headers = {"Authorization": f"Bearer {DEEPSEEK_API_KEY}", "Content-Type": "application/json"}
payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": prompt}],
    "temperature": 0.0,
}

resp = requests.post(url, headers=headers, json=payload, timeout=180)
resp.raise_for_status()
data = resp.json()

finish_reason = data["choices"][0].get("finish_reason", "")
content = data["choices"][0]["message"]["content"]

print("finish_reason:", finish_reason)
print("raw LLM output:", content)

def extract_cypher_from_llm(content: str) -> str:
    # 1) Try strict JSON
    try:
        obj = json.loads(content)
        if isinstance(obj, dict) and "cypher" in obj:
            return obj["cypher"].strip()
    except Exception:
        pass

    # 2) Try JSON substring (first { ... last })
    try:
        start = content.find("{")
        end = content.rfind("}")
        if start != -1 and end != -1 and end > start:
            obj = json.loads(content[start:end+1])
            if isinstance(obj, dict) and "cypher" in obj:
                return obj["cypher"].strip()
    except Exception:
        pass

    # 3) Try code fence
    m = re.search(r"```(?:cypher)?\s*([\s\S]*?)```", content, re.IGNORECASE)
    if m:
        return m.group(1).strip()

    # 4) Fallback: treat whole content as cypher
    return content.strip()

cypher = extract_cypher_from_llm(content)
print("--- Generated Cypher ---", cypher)


finish_reason: stop
raw LLM output: ```cypher
MATCH (encounter:Encounter)
WHERE 
  (($encounter_atCity_0 IS NOT NULL AND $encounter_atCity_0 IN ['city_paris', 'city_beauvais']) OR 
   ($encounter_atAirport_0 IS NOT NULL AND $encounter_atAirport_0 IN ['airport_cdg', 'airport_ory', 'airport_bva']))
OPTIONAL MATCH (encounter)-[:atAirport]->(airport:Airport)
OPTIONAL MATCH (encounter)-[:atCity]->(city:City)
OPTIONAL MATCH (airport)-[:locatedInCity]->(airportCity:City)
OPTIONAL MATCH (city)-[:locatedInCountry]->(country:Country)
OPTIONAL MATCH (airportCity)-[:locatedInCountry]->(airportCityCountry:Country)
OPTIONAL MATCH (encounter)-[:atCountry]->(encounterCountry:Country)
WITH encounter, airport, city, airportCity, country, airportCityCountry, encounterCountry
WHERE 
  ((city IS NOT NULL AND city.key IN ['city_paris', 'city_beauvais']) OR 
   (airport IS NOT NULL AND airport.key IN ['airport_cdg', 'airport_ory', 'airport_bva']))
MATCH (encounter)-[:hasDocumentCheck]->(docCheck:DocumentChec

In [34]:
print(prompt)

```txt
YOU ARE A CYPHER GENERATOR FOR GRAPHRAG RETRIEVAL.

GOAL
Create ONE Cypher query that returns triples relevant to the question:
“Which documents do I need to bring to passport control in Paris?”
The result set must be as close as possible to the reference logic.

OUTPUT FORMAT
Return rows with columns:
- s_label
- s_key
- p
- o_label
- o_key

HARD RULES
1. Use only: MATCH, OPTIONAL MATCH, WHERE, WITH, UNWIND, RETURN, LIMIT.
2. Output ONLY Cypher (no prose, no JSON).
3. Use UNWIND to emit triples as rows.
4. Always include LIMIT 2000.
5. Do NOT include Questioning/Stamping/Outcome/ScreeningStatus triples.
6. Include evidence + connection + scope triples only.

SCOPE RULES (PARIS)
Use ONLY these keys for Paris scope:
- Cities: city_paris, city_beauvais
- Airports: airport_cdg, airport_ory, airport_bva
Do NOT invent other keys or use fuzzy matching.

REQUIRED TRIPLE TYPES
Evidence:
- DocumentCheck -> requestedDocument -> DocumentInstance
- DocumentInstance -> documentType -> Litera

In [35]:
# 11 выполнить Cypher → получить triples.json в формате {s,p,o} + Цикл проверки Cypher на компил или нет

In [43]:
# === Cell 2: Execute Cypher and convert result rows -> triples JSON (s,p,o) ===

import json
from pathlib import Path
from neo4j import GraphDatabase
import requests

# ---- LLM helper (for fix) ----
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
if not DEEPSEEK_API_KEY:
    raise RuntimeError("DEEPSEEK_API_KEY is not set (load .env first).")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com").rstrip("/")
MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-reasoner")

prompt_2 = Path("prompt_2_reviewer.txt").read_text(encoding="utf-8")

def call_llm(prompt: str):
    url = f"{DEEPSEEK_BASE_URL}/v1/chat/completions"
    headers = {"Authorization": f"Bearer {DEEPSEEK_API_KEY}", "Content-Type": "application/json"}
    payload = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.0,
    }
    resp = requests.post(url, headers=headers, json=payload, timeout=180)
    resp.raise_for_status()
    data = resp.json()
    return data["choices"][0]["message"]["content"]


def extract_cypher_from_llm(content: str) -> str:
    # 1) Try strict JSON
    try:
        obj = json.loads(content)
        if isinstance(obj, dict) and "cypher" in obj:
            return obj["cypher"].strip()
    except Exception:
        pass

    # 2) Try JSON substring (first { ... last })
    try:
        start = content.find("{")
        end = content.rfind("}")
        if start != -1 and end != -1 and end > start:
            obj = json.loads(content[start:end+1])
            if isinstance(obj, dict) and "cypher" in obj:
                return obj["cypher"].strip()
    except Exception:
        pass

    # 3) Try code fence
    m = re.search(r"```(?:cypher)?\s*([\s\S]*?)```", content, re.IGNORECASE)
    if m:
        return m.group(1).strip()

    # 4) Fallback: treat whole content as cypher
    return content.strip()


def records_to_spo_triples(records):
    """
    Convert Neo4j rows to triples. If the query does not return the expected
    s_label/s_key/p/o_label/o_key columns, fall back to positional mapping or
    literal wrapping so the pipeline does not crash.
    """
    triples = []
    warned = False
    for r in records:
        if all(k in r for k in ["s_label", "s_key", "p", "o_label", "o_key"]):
            triples.append({
                "s": {"label": r["s_label"], "key": r["s_key"]},
                "p": r["p"],
                "o": {"label": r["o_label"], "key": r["o_key"]},
            })
            continue

        keys = list(r.keys())
        if not warned:
            print("WARNING: result rows do not have s_label/s_key/p/o_label/o_key. Falling back. Row keys:", keys)
            warned = True

        if len(keys) >= 5:
            triples.append({
                "s": {"label": r[keys[0]], "key": r[keys[1]]},
                "p": r[keys[2]],
                "o": {"label": r[keys[3]], "key": r[keys[4]]},
            })
        elif len(keys) == 1:
            k = keys[0]
            v = r[k]
            triples.append({
                "s": {"label": "Literal", "key": k},
                "p": "value",
                "o": {"label": "Literal", "key": str(v)},
            })
        else:
            triples.append({
                "s": {"label": "Literal", "key": "row"},
                "p": "value",
                "o": {"label": "Literal", "key": json.dumps(r, ensure_ascii=False)},
            })
    return triples

# ---- Execute Cypher ----

def run_cypher(cypher_text: str):
    driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
    try:
        with driver.session(database=DATABASE) as session:
            records = session.run(cypher_text).data()
        return records, None
    except Exception as e:
        return None, str(e)
    finally:
        driver.close()

records, exec_error = run_cypher(cypher)

# ---- If error, ask LLM to fix and retry once ----
if exec_error:
    print("CYPHER ERROR:", exec_error)
    fix_prompt = (
        f"{prompt_2}\n\n"
        f"QUESTION:\n{user_query}\n\n"
        f"DATABASE STRUCTURE:\n{schema_text}\n\n"
        f"CYPHER:\n{cypher}\n\n"
        f"ERROR:\n{exec_error}\n"
    )
    fix_out = call_llm(fix_prompt)
    cypher_fixed = extract_cypher_from_llm(fix_out)
    print("\n--- Fixed Cypher ---\n", cypher_fixed)
    records, exec_error = run_cypher(cypher_fixed)
    if exec_error:
        raise RuntimeError(f"Cypher still failing after fix: {exec_error}")
    cypher = cypher_fixed

retrieved_triples = records_to_spo_triples(records)

print(f"Retrieved rows: {len(records)}")
print("First 5 triples:")
print(json.dumps(retrieved_triples[:5], ensure_ascii=False, indent=2))


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `atCountry` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=14, column=22, offset=688>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 688, 'line': 14, 'column': 22}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Encounter)\nOPTIONAL MATCH (e)-[:atCity]->(scopeCity:City)\nOPTIONAL MATCH (e)-[:atAirport]->(scopeAirport:Airport)\nWITH DISTINCT e, scopeCity, scopeAirport\nWHERE scopeCity.key IN ['city_paris', 'city_beauvais']\n   OR scopeAirport.key IN ['airport_cdg', 'airport_ory', 'airport_bva']\nMATCH (e)-[:hasDoc

Retrieved rows: 24
First 5 triples:
[
  {
    "s": {
      "label": "Encounter",
      "key": "encounter_3902451_author_1"
    },
    "p": "hasDocumentCheck",
    "o": {
      "label": "DocumentCheck",
      "key": "doccheck_encounter_3902451_author_1_cash"
    }
  },
  {
    "s": {
      "label": "DocumentCheck",
      "key": "doccheck_encounter_3902451_author_1_cash"
    },
    "p": "requestedDocument",
    "o": {
      "label": "DocumentInstance",
      "key": "cash"
    }
  },
  {
    "s": {
      "label": "DocumentInstance",
      "key": "cash"
    },
    "p": "documentType",
    "o": {
      "label": "Literal",
      "key": "Cash"
    }
  },
  {
    "s": {
      "label": "Encounter",
      "key": "encounter_3902451_author_1"
    },
    "p": "atAirport",
    "o": {
      "label": "Airport",
      "key": "airport_cdg"
    }
  },
  {
    "s": {
      "label": "Airport",
      "key": "airport_cdg"
    },
    "p": "locatedInCity",
    "o": {
      "label": "City",
      "key": "city_p

In [ ]:
# 12 gold JSON → canonical strings, retrieved → canonical strings, Hits@k

In [40]:
import json
from pathlib import Path
import subprocess

def triple_to_canon_str(t):
    # case 1: already a string
    if isinstance(t, str):
        return t.strip()

    # case 2: dict in {s,p,o}
    if isinstance(t, dict) and "s" in t and "p" in t and "o" in t:
        return f"{t['s']['label']}|{t['s']['key']}::{t['p']}::{t['o']['label']}|{t['o']['key']}"

    raise TypeError(f"Unsupported triple format: {type(t)} | sample={repr(t)[:200]}")

# Collect ALL gold triples from annotation/q_1
q1_root = Path("/Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/annotation/q_74")
q1_files = list(q1_root.rglob("annotation_*.json"))

gold_triples = []
for p in q1_files:
    data = json.loads(p.read_text(encoding="utf-8"))
    if isinstance(data, dict) and "triples" in data:
        triples = data["triples"]
    elif isinstance(data, list):
        triples = data
    else:
        triples = []
    if isinstance(triples, list):
        gold_triples.extend(triples)

# Build canonical sets

gold_set = {triple_to_canon_str(t) for t in gold_triples}
retrieved_set = {triple_to_canon_str(t) for t in retrieved_triples}  # retrieved_triples from Cell 2

tp = len(gold_set & retrieved_set)
fp = len(retrieved_set - gold_set)
fn = len(gold_set - retrieved_set)

precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

print("Gold size:", len(gold_set))
print("Retrieved size:", len(retrieved_set))
print("TP:", tp, "FP:", fp, "FN:", fn)
print(f"Precision = {precision:.3f}")
print(f"Recall     = {recall:.3f}")
print(f"F1         = {f1:.3f}")

subprocess.run(["afplay", "/System/Library/Sounds/Blow.aiff"], check=False)


Gold size: 559
Retrieved size: 10
TP: 0 FP: 10 FN: 559
Precision = 0.000
Recall     = 0.000
F1         = 0.000


CompletedProcess(args=['afplay', '/System/Library/Sounds/Blow.aiff'], returncode=0)

In [13]:
print(gold_set)
print(retrieved_list)


{'DocumentInstance|documentinstance_encounter_12117026_author_1_foreign_passport::documentType::Literal|foreign_passport', 'DocumentInstance|documentinstance_encounter_11307373_author_1_foreign_passport::documentType::Literal|foreign_passport', 'Questioning|questioning_encounter_11265638_husband_1_accommodation_details::topicKey::Literal|accommodation_details', 'DocumentInstance|documentinstance_encounter_12002755_author_1_return_ticket::documentType::Literal|return_ticket', 'DocumentInstance|documentinstance_encounter_11483980_author_1_foreign_passport::documentType::Literal|foreign_passport', 'DocumentInstance|documentinstance_encounter_9900529_author_1_return_ticket::documentType::Literal|return_ticket', 'DocumentCheck|doccheck_encounter_11790790_friend_1_hotel_booking::requestedDocument::DocumentInstance|documentinstance_encounter_11790790_friend_1_hotel_booking', 'Encounter|encounter_12085432_author_1::hasDocumentCheck::DocumentCheck|doccheck_encounter_12085432_author_1_return_tic

NameError: name 'retrieved_list' is not defined

# Векторный retrieval

In [16]:
# 777
# Простой пример E5: эмбеддинги двух слов и расстояние между ними

import numpy as np
from sentence_transformers import SentenceTransformer

# Для русского языка удобен multilingual E5
model = SentenceTransformer("intfloat/multilingual-e5-small")

words = ["огромный", "огромный"]
inputs = [f"query: {w}" for w in words]  # для E5 нужен префикс
emb = model.encode(inputs, normalize_embeddings=True)

cosine_sim = float(np.dot(emb[0], emb[1]))
cosine_dist = 1.0 - cosine_sim
l2_dist = float(np.linalg.norm(emb[0] - emb[1]))

print(f"{words[0]} vs {words[1]}")
print(f"cosine_similarity = {cosine_sim:.4f}")
print(f"cosine_distance  = {cosine_dist:.4f}")
print(f"L2 distance      = {l2_dist:.4f}")


Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 14308.16it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


огромный vs огромный
cosine_similarity = 1.0000
cosine_distance  = -0.0000
L2 distance      = 0.0000


In [440]:
# 67876 - Vector retrieval pipeline (Encounters as index units)
from __future__ import annotations
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple
from neo4j import GraphDatabase
import os
import json

NEO4J_URI = "neo4j://localhost:7689"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "12345678"
NEO4J_DB = "neo4j"

EMBEDDING_PROP = "embedding_e5"
VECTOR_INDEX_NAME = "encounter_embedding_e5"
VECTOR_DIM = 768  # e5-base-v2

QUESTION = "Which airport do people most often fly into in Germany?"


def get_driver():
    return GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))


def ensure_vector_index(driver):
    with driver.session(database=NEO4J_DB) as session:
        session.run(
            f"""
            CREATE VECTOR INDEX {VECTOR_INDEX_NAME} IF NOT EXISTS
            FOR (e:Encounter) ON (e.{EMBEDDING_PROP})
            OPTIONS {{ indexConfig: {{ `vector.dimensions`: {VECTOR_DIM}, `vector.similarity_function`: 'cosine' }} }}
            """
        ).consume()


In [441]:
from collections import defaultdict

# Map smaller/metro airports to their larger city for embedding context
AIRPORT_ALIAS_CITY = {
    "airport_bva": "Paris",
    "airport_cdg": "Paris",
    "airport_ory": "Paris",
    "airport_sdg": "Paris",
    "airport_memmingen": "Munich",
    "airport_malpensa": "Milan",
    "airport_marco_polo": "Venice",
    "airport_fiumicino": "Rome",
    "airport_tegel": "Berlin",
    "airport_schonefeld": "Berlin",
    "airport_pisa": "Florence",
}

def fetch_encounter_cards(driver) -> Dict[str, str]:
    query = """
    MATCH (e:Encounter)
    OPTIONAL MATCH (e)-[:atCity]->(city:City)
    OPTIONAL MATCH (e)-[:atAirport]->(airport:Airport)
    OPTIONAL MATCH (e)-[:atCountry]->(country:Country)
    OPTIONAL MATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)-[:requestedDocument]->(di:DocumentInstance)
    OPTIONAL MATCH (e)-[:hasQuestioning]->(q:Questioning)
    RETURN e.key AS ekey,
           collect(DISTINCT city.key) AS cities,
           collect(DISTINCT airport.key) AS airports,
           collect(DISTINCT country.key) AS countries,
           collect(DISTINCT di.documentType) AS doc_types,
           collect(DISTINCT q.topicKey) AS topics
    """
    cards = {}
    with driver.session(database=NEO4J_DB) as session:
        for r in session.run(query):
            ekey = r["ekey"]
            if not ekey:
                continue
            parts = [f"Encounter: {ekey}"]
            if r["cities"]:
                parts.append("Cities: " + ", ".join(sorted([c for c in r["cities"] if c])))
            if r["airports"]:
                airports = sorted([a for a in r["airports"] if a])
                parts.append("Airports: " + ", ".join(airports))
                alias_cities = sorted({AIRPORT_ALIAS_CITY[a] for a in airports if a in AIRPORT_ALIAS_CITY})
                if alias_cities:
                    parts.append("AliasCities: " + ", ".join(alias_cities))
            if r["countries"]:
                parts.append("Countries: " + ", ".join(sorted([c for c in r["countries"] if c])))
            if r["doc_types"]:
                parts.append("Documents: " + ", ".join(sorted([d for d in r["doc_types"] if d])))
            if r["topics"]:
                parts.append("Topics: " + ", ".join(sorted([t for t in r["topics"] if t])))
            cards[ekey] = "".join(parts)
    return cards


In [442]:
# Table: vector-retrieved encounters -> city (dash if missing)
import pandas as pd

def vector_encounter_city_table(driver, question: str, k: int = 50):
    keys = retrieve_topk_encounters(driver, question, k=k)
    if not keys:
        return pd.DataFrame(columns=['encounter','city'])
    query = """
    MATCH (e:Encounter)
    WHERE e.key IN $keys
    OPTIONAL MATCH (e)-[:atCity]->(city:City)
    RETURN e.key AS encounter, coalesce(city.key, '-') AS city
    """
    with driver.session(database=NEO4J_DB) as session:
        rows = session.run(query, keys=keys)
        data = [dict(r) for r in rows]
    order = {k:i for i,k in enumerate(keys)}
    data.sort(key=lambda r: order.get(r['encounter'], 1e9))
    return pd.DataFrame(data)

with get_driver() as driver:
    df = vector_encounter_city_table(driver, QUESTION, k=50)
df


Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 11020.74it/s]
BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,encounter,city
0,encounter_11473534_mother_1,city_munich
1,encounter_11646006_author_1,city_munich
2,encounter_11320784_author_1,city_munich
3,encounter_11404518_author_1,city_munich
4,encounter_8841385_author_1,city_munich
5,encounter_11278346_author_1,city_memmingen
6,encounter_12052057_author_1,city_stuttgart
7,encounter_9611877_author_1,city_munich
8,encounter_9611949_author_1,city_munich
9,encounter_11280630_author_1,city_stuttgart


In [443]:
from typing import Iterable

try:
    from sentence_transformers import SentenceTransformer
except Exception as e:
    raise RuntimeError("Install sentence-transformers to compute E5 embeddings") from e


def embed_texts_e5(model: SentenceTransformer, texts: Iterable[str], is_query: bool = False) -> List[List[float]]:
    prefix = "query: " if is_query else "passage: "
    texts = [prefix + t for t in texts]
    vecs = model.encode(texts, normalize_embeddings=True)
    return [v.tolist() for v in vecs]


def build_embeddings_for_encounters(cards: Dict[str, str]):
    model = SentenceTransformer("intfloat/e5-base-v2")
    keys = list(cards.keys())
    texts = [cards[k] for k in keys]
    vectors = embed_texts_e5(model, texts, is_query=False)
    return keys, vectors


In [444]:
def write_embeddings(driver, keys: List[str], vectors: List[List[float]]):
    with driver.session(database=NEO4J_DB) as session:
        for k, v in zip(keys, vectors):
            session.run(
                "MATCH (e:Encounter {key:$k}) SET e.%s = $v" % EMBEDDING_PROP,
                k=k, v=v
            ).consume()


In [445]:
def retrieve_topk_encounters(driver, question: str, k: int = 50):
    model = SentenceTransformer("intfloat/e5-base-v2")
    q_vec = embed_texts_e5(model, [question], is_query=True)[0]
    with driver.session(database=NEO4J_DB) as session:
        result = session.run(
            f"""
            CALL db.index.vector.queryNodes('{VECTOR_INDEX_NAME}', $k, $vec)
            YIELD node, score
            RETURN node.key AS ekey, score
            ORDER BY score DESC
            """,
            k=k, vec=q_vec
        )
        return [r["ekey"] for r in result if r["ekey"]]


In [446]:
FILTER_PROMPT = """
Deterministic filter for: Which documents do I need to bring to passport control in France?
"""

from typing import Dict, Any

def llm_generate_filter(question: str) -> Dict[str, Any]:
    # Deterministic rule-based filter (q_21)
    return {
        "scope": {
            "cities": [],
            "airports": [],
            "countries": ["country_france"],
        },
        "evidence_predicates": [],
        "exclude_predicates": ["hasDocumentCheck", "requestedDocument", "documentType", "hasQuestioning", "hasStamping", "hasOutcome", "hasScreeningStatus"],
    }


In [447]:
USE_LLM_FILTER = True
def llm_generate_filter_deepseek(question, scope_catalog):
    """Generate filter JSON via DeepSeek."""
    api_key = os.environ.get('DEEPSEEK_API_KEY') or os.environ.get('DEEPSEEK_KEY')
    if not api_key:
        raise RuntimeError('Set DEEPSEEK_API_KEY env var')

    prompt = f"""
You are a filter generator for GraphRAG retrieval.
Return ONLY valid JSON.

QUESTION:
{question}

AVAILABLE SCOPE CATALOG (use ONLY these keys):
CITIES: {scope_catalog.get('cities', [])}
AIRPORTS: {scope_catalog.get('airports', [])}
COUNTRIES: {scope_catalog.get('countries', [])}

OUTPUT JSON SCHEMA:
{{
  "scope": {{"cities": [...], "airports": [...], "countries": [...] }},
  "evidence_predicates": [...],
  "exclude_predicates": [...]
}}

SCOPE RULES (ONLY THESE):
1) If the question mentions a specific city/airport/country, include ONLY those matching keys from the catalog.
2) If a scope list is EMPTY, it means "allow ALL" (do NOT invent keys).
3) Never enumerate all cities/airports/countries; use empty lists to mean ALL unless the question names specific ones.

EVIDENCE / EXCLUDE RULES:
- Choose evidence predicates to directly answer the question type:
  * Documents question → evidence_predicates include: hasDocumentCheck, requestedDocument, documentType
  * Question topics → hasQuestioning, questionTopic
  * Biometrics → hasBiometricCheck, biometricModality, biometricWasTaken
  * Screening → hasScreeningStatus
  * Airports/cities popularity → atAirport / atCity (keep in evidence_predicates)
- Exclude predicates that are irrelevant to the question. Use these defaults:
  * Always exclude: hasStamping, hasOutcome
  * If the question is NOT about docs: exclude hasDocumentCheck, requestedDocument, documentType
  * If NOT about questioning: exclude hasQuestioning, questionTopic
  * If NOT about biometrics: exclude hasBiometricCheck, biometricModality, biometricWasTaken

Return ONLY JSON.
"""

    payload = {
        'model': 'deepseek-reasoner',
        'messages': [
            {'role': 'system', 'content': 'You return ONLY valid JSON.'},
            {'role': 'user', 'content': prompt},
        ],
        'temperature': 0
    }

    resp = requests.post(
        'https://api.deepseek.com/v1/chat/completions',
        headers={'Authorization': f'Bearer {api_key}', 'Content-Type': 'application/json'},
        data=json.dumps(payload)
    )
    resp.raise_for_status()
    content = resp.json()['choices'][0]['message']['content']
    return json.loads(_extract_json(content))


In [448]:
from typing import List, Dict, Any

def apply_filter(triples: List[Dict[str, str]], f: Dict[str, Any]) -> List[Dict[str, str]]:
    cities = set(f.get('scope', {}).get('cities', []))
    airports = set(f.get('scope', {}).get('airports', []))
    countries = set(f.get('scope', {}).get('countries', []))
    evidence = set(f.get('evidence_predicates', []))
    exclude = set(f.get('exclude_predicates', []))

    def in_scope(val, allowed):
        return True if not allowed else val in allowed

    out = []
    for t in triples:
        if t['p'] in exclude:
            continue
        if t['p'] in evidence:
            out.append(t)
            continue

        if t['p'] == 'atCity' and in_scope(t['o_key'], cities):
            out.append(t)
        elif t['p'] == 'atAirport' and in_scope(t['o_key'], airports):
            out.append(t)
        elif t['p'] == 'atCountry' and in_scope(t['o_key'], countries):
            out.append(t)
        elif t['p'] == 'locatedInCity' and (in_scope(t['s_key'], airports) or in_scope(t['o_key'], cities)):
            out.append(t)
        elif t['p'] == 'locatedInCountry' and in_scope(t['o_key'], countries):
            out.append(t)
    return out


In [449]:
# Filter encounter keys by scope before fetching triples

def filter_encounter_keys_by_scope(driver, keys, scope):
    cities = scope.get('cities', [])
    airports = scope.get('airports', [])
    countries = scope.get('countries', [])

    has_cities = bool(cities)
    has_airports = bool(airports)
    has_countries = bool(countries)
    no_scope = not (has_cities or has_airports or has_countries)

    # If only country scope is provided, ignore vector top-k to avoid truncation
    if (not (has_cities or has_airports)) and has_countries:
        keys = []

    if no_scope and keys:
        return list(keys)

    query = '''
    MATCH (e:Encounter)
    OPTIONAL MATCH (e)-[:atCity]->(c:City)
    OPTIONAL MATCH (e)-[:atAirport]->(a:Airport)
    OPTIONAL MATCH (e)-[:atCountry]->(co:Country)

    OPTIONAL MATCH (c)-[:locatedInCountry]->(c_co:Country)
    OPTIONAL MATCH (a)-[:locatedInCity]->(a_city:City)
    OPTIONAL MATCH (a_city)-[:locatedInCountry]->(a_co:Country)

    WITH e, c, a, co, c_co, a_co
    WHERE (
        $no_scope = true
        OR ( ($has_cities = true) AND c.key IN $cities )
        OR ( ($has_airports = true) AND a.key IN $airports )
        OR ( ($has_countries = true) AND (
            co.key IN $countries OR
            c_co.key IN $countries OR
            a_co.key IN $countries
        ))
    )
    AND ( $keys = [] OR e.key IN $keys )

    RETURN DISTINCT e.key AS ekey
    '''
    with driver.session(database=NEO4J_DB) as session:
        rows = session.run(
            query,
            keys=keys or [],
            cities=cities,
            airports=airports,
            countries=countries,
            has_cities=has_cities,
            has_airports=has_airports,
            has_countries=has_countries,
            no_scope=no_scope,
        )
        return [r['ekey'] for r in rows]


In [450]:
# PIPELINE EXECUTION
with get_driver() as driver:
    ensure_vector_index(driver)
    cards = fetch_encounter_cards(driver)
    keys, vecs = build_embeddings_for_encounters(cards)
    write_embeddings(driver, keys, vecs)

    top_keys = retrieve_topk_encounters(driver, QUESTION, k=2000)
    if USE_LLM_FILTER:
        scope_catalog = fetch_scope_catalog(driver)
        flt = llm_generate_filter_deepseek(QUESTION, scope_catalog)
    else:
        flt = llm_generate_filter(QUESTION)
    scope = flt.get('scope', {})
    scoped_keys = filter_encounter_keys_by_scope(driver, top_keys, scope)
    raw_triples = fetch_triples_for_encounters(driver, scoped_keys)

filtered = apply_filter(raw_triples, flt)
retrieved_triples = filtered


Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 10041.83it/s]
BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 6365.18it/s]
BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical 

In [451]:
print(flt)

{'scope': {'cities': [], 'airports': [], 'countries': ['country_germany']}, 'evidence_predicates': ['atAirport'], 'exclude_predicates': ['hasStamping', 'hasOutcome', 'hasDocumentCheck', 'requestedDocument', 'documentType', 'hasQuestioning', 'questionTopic', 'hasBiometricCheck', 'biometricModality', 'biometricWasTaken', 'hasScreeningStatus']}


In [452]:
# Save retrieved triples to disk
from pathlib import Path
import json

out_path = Path("/Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/retrieval_outputs/retrieved_triples_q_74.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(filtered, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved", out_path)


Saved /Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/retrieval_outputs/retrieved_triples_q_74.json


In [453]:
# Metrics for current output
import json
from pathlib import Path

def triples_to_set_local(triples):
    s=set()
    for t in triples:
        if "s_label" in t:
            s.add((t["s_label"], t["s_key"], t["p"], t["o_label"], t["o_key"]))
        else:
            s_node=t.get("s", {})
            o_node=t.get("o", {})
            s.add((s_node.get("label"), s_node.get("key"), t.get("p"), o_node.get("label"), o_node.get("key")))
    return s

def load_q1_gold_local():
    base = Path("/Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/annotation/q_74")
    triples = []
    for p in base.rglob("annotation_*.json"):
        data = json.loads(p.read_text(encoding="utf-8"))
        triples.extend(data.get("triples", []))
    return triples

retrieved_set = triples_to_set_local(retrieved_triples)
gold_set = triples_to_set_local(load_q1_gold_local())
tp = len(retrieved_set & gold_set)
fp = len(retrieved_set - gold_set)
fn = len(gold_set - retrieved_set)
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
print("Gold size:", len(gold_set))
print("Retrieved size:", len(retrieved_set))
print("TP:", tp, "FP:", fp, "FN:", fn)
print(f"Precision = {precision:.3f}")
print(f"Recall     = {recall:.3f}")
print(f"F1         = {f1:.3f}")
import subprocess
subprocess.run(["afplay", "/System/Library/Sounds/Blow.aiff"], check=False)


Gold size: 537
Retrieved size: 536
TP: 536 FP: 0 FN: 1
Precision = 1.000
Recall     = 0.998
F1         = 0.999


CompletedProcess(args=['afplay', '/System/Library/Sounds/Blow.aiff'], returncode=0)

# Запись в базу уже из файла с диска 

In [314]:
from __future__ import annotations
from pathlib import Path
from typing import Optional, Iterable
from neo4j import GraphDatabase


def _split_cypher_statements(text: str) -> list[str]:
    """
    Split on semicolons, ignoring empty chunks.
    (Assumes your generator doesn't put ';' inside string literals.)
    """
    parts = [p.strip() for p in text.split(";")]
    return [p for p in parts if p]


def run_cypher_file(
    cypher_path: str,
    uri: str = "neo4j://localhost:7688",
    user: str = "neo4j",
    password: str = "12345678",
    database: Optional[str] = None,
) -> None:
    """
    Reads Cypher statements from a file and executes them sequentially.
    Each statement is run as its own query (Neo4j requires 1 statement per run()).
    """
    path = Path(cypher_path)
    if not path.exists():
        raise FileNotFoundError(f"Cypher file not found: {path}")

    cypher_text = path.read_text(encoding="utf-8")
    statements = _split_cypher_statements(cypher_text)

    if not statements:
        print("No Cypher statements found in file.")
        return

    driver = GraphDatabase.driver(uri, auth=(user, password))
    try:
        with driver.session(database=database) as session:
            for i, stmt in enumerate(statements, start=1):
                session.run(stmt).consume()  # consume forces execution now
                # print(f"✅ {i}/{len(statements)} executed")
        print(f"✅ Executed {len(statements)} statements from {path.name}")
    finally:
        driver.close()

# run_cypher_file("cypher_checkpoints/cypher_1.txt", uri="neo4j://localhost:7688", user="neo4j", password="12345678")

# Очистить базу 2 (secondary) и внести 1 cypher

In [37]:
step_1_reset_db_2()

import time
time.sleep(9)

first = 1
last = 59
for i in range(first, last + 1):  # от 1 до посл-1 включительно
    run_cypher_file(
        f"cypher_checkpoints/fr_1/cypher_{i}.txt",
        uri="neo4j://localhost:7689",
        user="neo4j",
        password="12345678"
    )
first = 1
last = 57
for i in range(first, last + 1):  # от 1 до посл-1 включительно
    run_cypher_file(
        f"cypher_checkpoints/fr_2/cypher_{i}.txt",
        uri="neo4j://localhost:7689",
        user="neo4j",
        password="12345678"
    )
# first = 1
# last = 210
# for i in range(first, last + 1):  # от 1 до посл-1 включительно
#     run_cypher_file(
#         f"cypher_checkpoints/germany/cypher_{i}.txt",
#         uri="neo4j://localhost:7689",
#         user="neo4j",
#         password="12345678"
#     )
# first = 1
# last = 87
# for i in range(first, last + 1):  # от 1 до посл-1 включительно
#     run_cypher_file(
#         f"cypher_checkpoints/italy/cypher_{i}.txt",
#         uri="neo4j://localhost:7689",
#         user="neo4j",
#         password="12345678"
#     )
# first = 1
# last = 23
# for i in range(first, last + 1):  # от 1 до посл-1 включительно
#     run_cypher_file(
#         f"cypher_checkpoints/spain/cypher_{i}.txt",
#         uri="neo4j://localhost:7689",
#         user="neo4j",
#         password="12345678"
#     )

# a = 3
# for i in range(a, a + 1):  # от 1 до 11 включительно
#     run_cypher_file(
#         f"cypher_checkpoints/cypher_{i}.txt",
#         uri="neo4j://localhost:7689",
#         user="neo4j",
#         password="12345678"
#     )
import subprocess
subprocess.run(["afplay", "/System/Library/Sounds/Blow.aiff"], check=False)


neo4j_travel2
neo4j_travel2
neo4j_travel2_data
7e0a20fc4393b3909afe057afc391d420a303a8c0ebda5278f31f8758b5a3458
Did reset DB #2 (http://localhost:7476, bolt neo4j://localhost:7689)
✅ Executed 69 statements from cypher_1.txt
✅ Executed 35 statements from cypher_2.txt
✅ Executed 44 statements from cypher_3.txt
✅ Executed 22 statements from cypher_4.txt
✅ Executed 55 statements from cypher_5.txt
✅ Executed 18 statements from cypher_6.txt
✅ Executed 44 statements from cypher_7.txt
✅ Executed 25 statements from cypher_8.txt
✅ Executed 22 statements from cypher_9.txt
✅ Executed 25 statements from cypher_10.txt
✅ Executed 29 statements from cypher_11.txt
✅ Executed 18 statements from cypher_12.txt
✅ Executed 38 statements from cypher_13.txt
✅ Executed 29 statements from cypher_14.txt
✅ Executed 21 statements from cypher_15.txt
✅ Executed 27 statements from cypher_16.txt
✅ Executed 18 statements from cypher_17.txt
✅ Executed 36 statements from cypher_18.txt
✅ Executed 30 statements from cypher

CompletedProcess(args=['afplay', '/System/Library/Sounds/Blow.aiff'], returncode=0)

In [39]:


from neo4j import GraphDatabase

URI = "neo4j://localhost:7688"
USER = "neo4j"
PASSWORD = "12345678"
DATABASE = "neo4j"

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

try:
    with driver.session(database=DATABASE) as session:
        # Node labels
        labels = session.run("CALL db.labels()").value()
        print("NODE LABELS:")
        for l in sorted(labels):
            print(f"- {l}")

        # Relationship types
        rels = session.run("CALL db.relationshipTypes()").value()
        print("\nRELATIONSHIP TYPES:")
        for r in sorted(rels):
            print(f"- {r}")

        # Property keys
        props = session.run("CALL db.propertyKeys()").value()
        print("\nPROPERTY KEYS:")
        for p in sorted(props):
            print(f"- {p}")
finally:
    driver.close()


NODE LABELS:
- AdditionalChecks
- Airport
- AtDesk
- BorderOfficer
- Cash
- CashAmount
- City
- Country
- DocumentCheck
- Employment
- Encounter
- HotelBookingDoc
- HotelPaymentCard
- HotelPaymentStatus
- Outcome
- PaymentCard
- Questioning
- ReturnTicket
- Stamping
- TravelInsurance
- Traveller
- VisitDuration
- VisitPurpose

RELATIONSHIP TYPES:
- atAirport
- atCity
- controlStage
- hasDocumentCheck
- hasOfficer
- hasOutcome
- hasQuestioning
- hasScreeningStatus
- hasStamping
- hasTraveler
- locatedInCity
- locatedInCountry
- questionTopic
- requestedDocType

PROPERTY KEYS:
- controlStage
- encounterDurationSec
- entityName
- key
- presented
- sourcePostAuthor
- sourcePostDateTime
- sourcePostID
- sourcePostURL
- wasAsked
